# 02 — Method Comparison

Run the full benchmark pipeline across composition, k-mer TF-IDF, and ESM-2 embedders.
Compare using trustworthiness and 5-NN CV accuracy.

In [ ]:
import logging
from pathlib import Path
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')

from prot2vec.data.pfam import download_pfam_seed, parse_pfam_families
from prot2vec.data.dataset import ProteinDataset

PFAM_IDS = ["PF00069", "PF00072"]

seed_path = download_pfam_seed(version="35.0", cache_dir="../data/raw")
records = parse_pfam_families(PFAM_IDS, seed_path)
dataset = ProteinDataset.from_pfam_records(records)
print(dataset)

In [ ]:
from prot2vec.embedders.composition import CompositionEmbedder
from prot2vec.embedders.kmer import KmerEmbedder
from prot2vec.reduction.reducers import UMAPReducer
from prot2vec.pipeline import RunConfig, run

config = RunConfig(
    dataset=dataset,
    embedders=[
        CompositionEmbedder(),
        KmerEmbedder(k=3),
    ],
    reducer=UMAPReducer(n_neighbors=15, min_dist=0.1, metric="cosine"),
    results_dir=Path("../results"),
    cache_embeddings=True,
    save_figures=True,
)

results = run(config)
results

In [ ]:
from prot2vec.visualization.plots import metrics_bar_chart

metrics_bar_chart(results, metric="trustworthiness", title="Trustworthiness by Method")
metrics_bar_chart(results, metric="knn_accuracy_mean", title="5-NN CV Accuracy by Method")